In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

root_dir = Path.cwd().resolve()
data_path = (root_dir / ".." / "task1" / "data.csv").resolve()
model_path = (root_dir / ".." / "task1" / "spam_classifier.joblib").resolve()
output_dir = (root_dir / "outputs").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

print("Data:", data_path)
print("Model:", model_path)
print("Outputs:", output_dir)

In [ ]:
df = pd.read_csv(data_path)

for col in df.columns:
    if "Unnamed" in str(col):
        df = df.drop(columns=[col])

df = df.dropna(axis=1, how="all")

label_col = "v1" if "v1" in df.columns else df.columns[0]
text_col = "v2" if "v2" in df.columns else df.columns[1]

df[label_col] = df[label_col].astype(str)
df[text_col] = df[text_col].astype(str)

X_text = df[text_col]
y_true = df[label_col]

model = joblib.load(model_path)

print("Columns:", df.columns.tolist())
print("Label column:", label_col)
print("Text column:", text_col)
print("Model type:", type(model))
print("Class labels:", getattr(model, "classes_", None))

In [ ]:
sensitive_cols = []

if sensitive_cols:
    print("Sensitive columns:", sensitive_cols)
    for col in sensitive_cols:
        print(col, "groups:", df[col].dropna().unique()[:10])
else:
    print("No sensitive attributes found; fairness checks by group will be skipped.")

In [ ]:
from sklearn.inspection import permutation_importance

vectorizer = None
estimator = model

if hasattr(model, "named_steps"):
    for step in model.named_steps.values():
        if hasattr(step, "get_feature_names_out"):
            vectorizer = step
    estimator = list(model.named_steps.values())[-1]

feature_names = None
if vectorizer is not None:
    feature_names = vectorizer.get_feature_names_out()

importance = None
if hasattr(estimator, "coef_"):
    importance = np.abs(estimator.coef_).ravel()
elif hasattr(estimator, "feature_importances_"):
    importance = estimator.feature_importances_
else:
    if vectorizer is None:
        raise ValueError("Permutation importance needs a vectorized feature matrix.")
    X_vec = vectorizer.transform(X_text)
    perm = permutation_importance(estimator, X_vec, y_true, n_repeats=3, random_state=42)
    importance = perm.importances_mean

if feature_names is None:
    feature_names = np.arange(len(importance)).astype(str)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values("importance", ascending=False)

importance_df.head(20)

In [ ]:
from lime.lime_text import LimeTextExplainer

if not hasattr(model, "predict_proba"):
    raise ValueError("LIME needs predict_proba for class probabilities.")

class_names = [str(c) for c in getattr(model, "classes_", ["ham", "spam"])]
explainer = LimeTextExplainer(class_names=class_names)

predict_proba = model.predict_proba

spam_row = df[df[label_col].str.lower() == "spam"]
ham_row = df[df[label_col].str.lower() == "ham"]

samples = []
if not spam_row.empty:
    samples.append(spam_row.iloc[0][text_col])
if not ham_row.empty:
    samples.append(ham_row.iloc[0][text_col])
if not samples:
    samples = [df.iloc[0][text_col]]

for i, text in enumerate(samples, start=1):
    exp = explainer.explain_instance(text, predict_proba, num_features=10)
    print(f"Example {i} text:", text[:120], "...")
    display(pd.DataFrame(exp.as_list(), columns=["feature", "weight"]))

In [ ]:
from sklearn.metrics import confusion_matrix

if not sensitive_cols:
    print("No sensitive attributes available; fairness metrics by group are not computed.")
else:
    y_pred = model.predict(X_text)
    pos_label = "spam" if "spam" in set(y_true.str.lower()) else y_true.unique()[0]

    for col in sensitive_cols:
        print("\nGroup metrics for:", col)
        for group, group_df in df.groupby(col):
            y_g = group_df[label_col].astype(str)
            X_g = group_df[text_col].astype(str)
            y_p = model.predict(X_g)

            tn, fp, fn, tp = confusion_matrix(
                y_g.str.lower() == pos_label,
                y_p.astype(str).str.lower() == pos_label,
                labels=[False, True]
            ).ravel()

            selection_rate = (tp + fp) / max((tp + fp + tn + fn), 1)
            tpr = tp / max((tp + fn), 1)
            fpr = fp / max((fp + tn), 1)

            print(
                f"Group={group} | selection_rate={selection_rate:.3f} | "
                f"TPR={tpr:.3f} | FPR={fpr:.3f}"
            )